Libraries

In [ ]:
import pandas as pd
import requests
from pathlib import Path
import time
import re
import shutil

Now we create the necessary paths for saving the .FITS and reading the catalog
- The catalog must be previosly filtered
- The file for the retries, is going to be saved in the same folder as the .FITS

In [ ]:
BASE_DIR = Path(__file__).resolve().parent.parent
EXCEL_FILE = BASE_DIR / "scripts" / "catalog.xlsx" 
OUTPUT_BASE = BASE_DIR / "data" / "IUE_FITS" 
RETRY_FILE = OUTPUT_BASE / "retries.txt"
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

print('All paths settled')

Catalog reading and base URL for downloads
- We use 'zfill' because some images start with "0" and excel deletes the zero by default
- The URL for each image is constructed as follows: 
BASE_URL?filename=Cam+Image+Product.FITS

- Product is the sufix for the FITS files, there are many, but for us, is enough with RL

In [ ]:
df = pd.read_excel(EXCEL_FILE)
df["Cam"] = df["Cam"].astype(str).str.strip()
df["Image"] = df["Image"].astype(str).str.zfill(5) 

BASE_URL = "http://ines.ts.astro.it/cgi-ines/SingleDownload"
PRODUCTS = ["RL"]

print('Catalog loaded and URL settled')

Due to bad internet connection or servers saturation, is good to set a waiting time between files, between retries and a max amount of retries per file. We are going to set a session to ask the server for files, (academic use)

In [ ]:
MAX_RETRIES = 3
RETRY_SLEEP = 5
BETWEEN_FILES = 1.5

session = requests.Session()
session.headers.update(
    {"User-Agent": "IUE-bulk-downloader (academic use)"}
)

The following functions go though the download process, downloading the .FITS files. In case of failure after the max_retries, the image is added to the retry file, for later processing. 

In [ ]:
def is_fits_response(response):
    ct = response.headers.get("Content-Type", "").lower()
    return response.status_code == 200 and not ct.startswith("text")


def download_product(cam, image, product, outdir):
    filename = f"{cam}{image}{product}.FITS"
    url = f"{BASE_URL}?filename={filename}"
    outfile = outdir / filename

    if outfile.exists():
        return True  # success

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = session.get(url, timeout=90)
            if is_fits_response(r):
                with open(outfile, "wb") as f:
                    f.write(r.content)
                time.sleep(BETWEEN_FILES)
                return True
            else:
                return False

        except requests.exceptions.RequestException:
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_SLEEP)
            else:
                return False


def process_image(cam, image):
    image_id = f"{cam}{image}"
    print(f"\n⬇ Processing {image_id}")

    image_dir = OUTPUT_BASE / image_id
    image_dir.mkdir(exist_ok=True)

    downloaded_any = False

    for product in PRODUCTS:
        success = download_product(cam, image, product, image_dir)
        if success:
            downloaded_any = True

    if downloaded_any:
        print("✅ Success")
    else:
        print("❌ Fail")

    return downloaded_any

Now we are going to build the main loop to download all files. This next chunk of code may take more than 1 hour to complete. 

In [ ]:
retries = []
for _, row in df.iterrows():
    ok = process_image(row["Cam"], row["Image"])
    if not ok:
        retries.append(f"{row['Cam']}{row['Image']}")

Sometimes we can get a "fake download", where the folder for the file is created but nothing is downloades, the code marks these cases as success. To fix this we are going to find this empty folders, delete them and add the image to the retries file.

In [ ]:
def cleaning(base_path):

    extra_retries = []
    
    for folder in base_path.iterdir():
        if folder.is_dir():
            files = list(folder.glob("*"))
            
            if not files:
                print(f"Fake download: {folder.name}. Deleting folder...")
                extra_retries.append(folder.name)
                shutil.rmtree(folder)
                
    return extra_retries


Finally for this first attempt we store the names of the images that were not dowloaded in a *.txt file. 

In [ ]:
fake_downloads = cleaning(OUTPUT_BASE)
failed_images = retries + fake_downloads

if failed_images:
    with open(RETRY_FILE, "w") as f:
        for img in failed_images:
            f.write(img + "\n")

    print(
        f"\n{len(failed_images)} failed images."
        f"\nSaved in: {RETRY_FILE}"
    )
else:
    print("\n All .FITS downloaded successfully.")

In case we do have files to retry: 

In [ ]:
retries = []

print(f"Reading IDs from {RETRY_FILE}...")

with open(RETRY_FILE, "r") as f:
    ids_to_retry = [line.strip() for line in f.readlines() if line.strip()]

for image_full_id in ids_to_retry:
    match = re.match(r"([a-zA-Z]+)([0-9]+)", image_full_id)
    if match:
        cam = match.group(1).upper()
        image = match.group(2).zfill(5)
        
        ok = process_image(cam, image)
        if not ok:
            retries.append(image_full_id)
    else:
        print(f"unrecognized ID: {image_full_id}")

fake_downloads = cleaning(OUTPUT_BASE)
failed_images = retries + fake_downloads

if failed_images:
    with open(RETRY_FILE, "w") as f:
        for img in failed_images:
            f.write(img + "\n")
    print(f"\n{len(failed_images)} images still failing, retry file has been updated. RUN THIS CELL AGAIN")
else:
    RETRY_FILE.write_text("") 
    print("All fits downloaded successfully")

If after a few iterations we still have failed cases, check for the manual download at the INES catalog: https://sdc.cab.inta-csic.es/ines/index2.html since it may not exist